# Figure 6 -- Analog generation

In [ ]:
import pandas as pd,numpy as np,matplotlib,matplotlib.pyplot as plt,matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm,LinearSegmentedColormap,TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle
import colorsys,seaborn as sns,warnings
warnings.filterwarnings('ignore')

matplotlib.font_manager._load_fontmanager(try_read_cache=False)
plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'font.size':6,'axes.titlesize':6,'axes.labelsize':6,
    'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':6,
    'axes.linewidth':0.6,'axes.edgecolor':'#333333',
    'xtick.color':'#333333','ytick.color':'#333333',
    'axes.labelcolor':'#222222','font.family':'sans-serif',
    'font.sans-serif':['Arial','Liberation Sans','DejaVu Sans'],
    'figure.dpi':300,'savefig.dpi':600,
    'xtick.major.width':0.5,'ytick.major.width':0.5,
    'xtick.major.size':2.5,'ytick.major.size':2.5,
    'xtick.major.pad':2,'ytick.major.pad':2,
    'axes.spines.top':False,'axes.spines.right':False,
})

DATA='/home/pszymczak/code/omegamp-dashboard/data/'
ref=pd.read_csv(DATA+'omegamp_reference_table.csv')
mic=pd.read_csv(DATA+'mic.csv'); hc50=pd.read_csv(DATA+'hc50.csv')
npn=pd.read_csv(DATA+'npn.csv'); disc=pd.read_csv(DATA+'disc.csv')
bestsel=pd.read_csv(DATA+'bestsel.csv'); cc50=pd.read_csv(DATA+'cc50.csv')
try:
    rr=pd.read_csv(DATA+'reference_amps.csv'); ref_amps_seq=rr[rr['IsAMP']==1]['Sequence'].tolist()
except FileNotFoundError:
    ref_amps_seq=ref['sequence'].dropna().tolist()
HYDRO_SCALE={'A':0.62,'R':-2.53,'N':-0.78,'D':-0.90,'C':0.29,'Q':-0.85,'E':-0.74,
             'G':0.48,'H':-0.40,'I':1.38,'L':1.06,'K':-1.50,'M':0.64,'F':1.19,
             'P':0.12,'S':-0.18,'T':-0.05,'W':0.81,'Y':0.26,'V':1.08}
CHARGE_MAP={'K':1,'R':1,'H':0.5,'D':-1,'E':-1}
ref_charges=[sum(CHARGE_MAP.get(aa,0) for aa in s.upper()) for s in ref_amps_seq]
ref_hydros=[np.mean([HYDRO_SCALE.get(aa,0) for aa in s.upper()]) for s in ref_amps_seq]
ref_amp_props=pd.DataFrame({'charge':ref_charges,'hydrophobicity':ref_hydros})

df=ref.merge(hc50,on='short_name',how='left').merge(cc50,on='short_name',how='left')
df=df.merge(npn.rename(columns={'MaxRel':'NPN','AUC':'NPN_AUC'}),on='short_name',how='left')
df=df.merge(disc.rename(columns={'MaxRel':'DiSC','AUC':'DiSC_AUC'}),on='short_name',how='left')
mic_strains=mic.columns[1:]; mic_vals=mic[mic_strains].astype(float)
mic['n_strains_le2']=(mic_vals<=2).sum(axis=1); mic['n_strains_le4']=(mic_vals<=4).sum(axis=1)
mic['mic50']=mic_vals.fillna(128).clip(upper=128).median(axis=1)
df=df.merge(mic[['short_name','n_strains_le2','n_strains_le4','mic50']],on='short_name',how='left')
df['HC50']=df['HC50'].clip(0.1,128); df['CC50']=df['CC50'].clip(0.1,128)
df['SW']=(df['HC50']/df['mic50']).clip(0.01,1e6)

proto_map={'DBAASPS_20015':'OP-145-TII4','DBAASPS_20955':'As-CATH4-6L'}
def get_family(row):
    if row['category']=='prototype':
        name=row['short_name']
        for fam in ['BoCo1','GQ20','Mammutin-1','DeNo1047','cecropin','LG21','pa4','sarcotoxin','bZIP']:
            if fam.lower() in name.lower() or fam==name: return fam
        if 'OP-145' in name: return 'OP-145-TII4'
        if 'As-CATH' in name: return 'As-CATH4-6L'
        return name
    p=str(row.get('prototype','')); return proto_map.get(p,p) if p!='nan' else ''
df['family']=df.apply(get_family,axis=1)
proto_by_fam={}
for _,row in df[df['category']=='prototype'].iterrows(): proto_by_fam[row['family']]=row
antimicrobial=df[(df['category']=='analog')&(df['objective']=='antimicrobial')].copy()

FC={'Mammutin-1':'#D55E00','GQ20':'#E69F00','BoCo1':'#009E73',
    'DeNo1047':'#56B4E9','As-CATH4-6L':'#0072B2','OP-145-TII4':'#CC79A7'}
INACTIVE_ORDER=['As-CATH4-6L','Mammutin-1','BoCo1','GQ20','DeNo1047','OP-145-TII4']

# Short labels for scatter annotations (Omega-num only)
LEADS_SHORT={
    '\u03a9-AT-BoCo1-5':'\u03a9-5', '\u03a9-AT-BoCo1-9':'\u03a9-9',
    '\u03a9-AU-GQ20-4':'\u03a9-4', '\u03a9-AP-Mammutin-1-4':'\u03a9-4',
}
# Full labels for heatmap E y-tick
LEADS_FULL={
    '\u03a9-AT-BoCo1-5':'\u03a9-AT-BoCo1-5', '\u03a9-AT-BoCo1-9':'\u03a9-AT-BoCo1-9',
    '\u03a9-AU-GQ20-4':'\u03a9-AU-GQ20-4', '\u03a9-AP-Mammutin-1-4':'\u03a9-AP-Mammutin-1-4',
}
LEADS_SET=set(LEADS_SHORT.keys())

def annotate_lead(ax,x,y,label,offset=(12,8)):
    ax.annotate(label,(x,y),fontsize=6,ha='left',va='center',color='black',
                xytext=offset,textcoords='offset points',
                arrowprops=dict(arrowstyle='-',color='#555',lw=0.5),zorder=10)

STRAIN_LABELS={
    'A. baumannii ATCC 19606 (-)':'$\\it{A. baumannii}$ ATCC 19606',
    'A. baumannii ATCC BAA-1605 (-) - CGTPACCIMRA':'$\\bf{\\it{A. baumannii}}$ $\\bf{BAA\\text{-}1605\\;(CRAB)}$',
    'E. cloacae ATCC 13047 (-)':'$\\it{E. cloacae}$ ATCC 13047',
    'E. coli ATCC 11775 (-)':'$\\it{E. coli}$ ATCC 11775',
    'E. coli AIC221 (-)':'$\\it{E. coli}$ AIC221',
    'E. coli AIC222 - CRE (-)':'$\\bf{\\it{E. coli}}$ $\\bf{AIC222\\;(CRE)}$',
    'E. coli ATCC BAA-3170 (-) - CRE':'$\\bf{\\it{E. coli}}$ $\\bf{BAA\\text{-}3170\\;(CRE)}$',
    'K. pneumoniae ATCC 13883 (-)':'$\\it{K. pneumoniae}$ ATCC 13883',
    'K. pneumoniae ATCC BAA-2342 (-) - EIRK':'$\\bf{\\it{K. pneumoniae}}$ $\\bf{BAA\\text{-}2342\\;(ESBL)}$',
    'P. aeruginosa PAO1 (-)':'$\\it{P. aeruginosa}$ PAO1',
    'P. aeruginosa PA14 (-)':'$\\it{P. aeruginosa}$ PA14',
    'P. aeruginosa ATCC BAA-3197 (-) - FBCRP':'$\\bf{\\it{P. aeruginosa}}$ $\\bf{BAA\\text{-}3197\\;(FQR)}$',
    'S. enterica ATCC 9150 (-)':'$\\it{S. enterica}$ ATCC 9150',
    'S. enterica Typhimurtium ATCC 700720':'$\\it{S. enterica}$ Typhimurium',
    'B. subtilis ATCC 23857 (+)':'$\\it{B. subtilis}$ ATCC 23857',
    'S. aureus ATCC 12600 (+)':'$\\it{S. aureus}$ ATCC 12600',
    'S. aureus ATCC BAA-1556 - MRSA (+)':'$\\bf{\\it{S. aureus}}$ $\\bf{BAA\\text{-}1556\\;(MRSA)}$',
    'E. faecalis ATCC 700802 - VRE (+)':'$\\bf{\\it{E. faecalis}}$ $\\bf{700802\\;(VRE)}$',
    'E. faecium ATCC 700221 - VRE (+)':'$\\bf{\\it{E. faecium}}$ $\\bf{700221\\;(VRE)}$',
    'E. coli K-12 BW25113 (-)':'$\\it{E. coli}$ K-12 BW25113',
}
strain_order=[]
ecoli_strains=[s for s in mic_strains if 'E. coli' in s or 'coli' in s.lower()]
for s in mic_strains:
    if s in ecoli_strains: continue
    strain_order.append(s)
    if 'cloacae' in s.lower(): strain_order.extend(ecoli_strains)
if not any('coli' in s.lower() for s in strain_order): strain_order.extend(ecoli_strains)
STRAIN_MDR={s:any(t in s for t in ['CRAB','CRE','ESBL','EIRK','FBCR','FQR','MRSA','VRE','CGTPA']) for s in mic_strains}
STRAIN_GRAM={s:('+' if '(+)' in s else '-') for s in mic_strains}
print(f"Antimicrobial analogs: {len(antimicrobial)}")

# ===================== FIGURE =====================
fig=plt.figure(figsize=(6.5,8.0),dpi=300)
gs_ac=gridspec.GridSpec(1,2,figure=fig,left=0.10,right=0.92,top=0.955,bottom=0.845,wspace=0.55)
gs_d=gridspec.GridSpec(1,6,figure=fig,left=0.04,right=0.91,top=0.775,bottom=0.610,wspace=0.08)
gs_e=gridspec.GridSpec(1,2,figure=fig,left=0.12,right=0.97,top=0.560,bottom=0.395,wspace=0.35)
gs_b=gridspec.GridSpec(1,1,figure=fig,left=0.10,right=0.76,top=0.325,bottom=0.200,wspace=0.3)
L=dict(fontsize=8,fontweight='bold',va='top',ha='left')
sw_norm=LogNorm(vmin=0.1,vmax=10000); sw_cmap='RdYlGn'

fig.legend(
    [Line2D([],[],marker='D',color='w',markerfacecolor='#888',markeredgecolor='none',ms=4,lw=0),
     Line2D([],[],marker='o',color='w',markerfacecolor='#aaa',markeredgecolor='none',ms=3.5,lw=0)],
    ['Prototype','Analog'],fontsize=6,loc='upper center',bbox_to_anchor=(0.50,0.998),
    frameon=True,framealpha=0.95,edgecolor='#ccc',ncol=2,handletextpad=0.3,columnspacing=1.2)

# A
ax=fig.add_subplot(gs_ac[0]); ax.text(-0.20,1.15,'A',transform=ax.transAxes,**L)
for i,fam in enumerate(INACTIVE_ORDER):
    proto=proto_by_fam.get(fam); ana=antimicrobial[antimicrobial['family']==fam]; color=FC[fam]
    p_str=proto['n_strains_le2'] if proto is not None and pd.notna(proto['n_strains_le2']) else 0
    ax.scatter(p_str,i,marker='D',c=color,edgecolors='none',s=18,zorder=5)
    if len(ana)>0:
        jitter=np.random.RandomState(42+i).normal(0,0.10,len(ana))
        for j,(idx,row) in enumerate(ana.iterrows()):
            if row['short_name'] in LEADS_SET:
                ax.scatter(row['n_strains_le2'],i+jitter[j],c=color,edgecolors='black',linewidth=0.5,s=14,marker='o',zorder=5)
                annotate_lead(ax,row['n_strains_le2'],i+jitter[j],LEADS_SHORT[row['short_name']],(8,7))
            else:
                ax.scatter(row['n_strains_le2'],i+jitter[j],c=color,edgecolors='none',linewidth=0.2,s=10,marker='o',alpha=0.9,zorder=4)
        active=(ana['n_strains_le4']>=1).sum()
        ax.annotate(f'{active}/{len(ana)}',xy=(1.02,i),xycoords=('axes fraction','data'),fontsize=6,va='center',ha='left',color='#555',annotation_clip=False)
ax.set_xlabel('Strains with MIC $\\leq$ 2 $\\mu$M',fontsize=6)
ax.set_yticks(range(len(INACTIVE_ORDER))); ylabels=ax.set_yticklabels(INACTIVE_ORDER,fontsize=6)
for yl,fam in zip(ylabels,INACTIVE_ORDER): yl.set_color(FC[fam])
ax.invert_yaxis(); ax.set_xlim(-0.5,14.5)

# B
ax=fig.add_subplot(gs_ac[1]); ax.text(-0.12,1.15,'B',transform=ax.transAxes,**L)
bs=bestsel.merge(ref[['short_name','category','prototype']],on='short_name',how='left')
bs['family']=bs.apply(get_family,axis=1)
for i,fam in enumerate(INACTIVE_ORDER):
    color=FC[fam]; proto_bs=bs[(bs['family']==fam)&(bs['category']=='prototype')]
    ana_bs=bs[(bs['family']==fam)&(bs['category']!='prototype')]
    if len(proto_bs)>0: ax.scatter(i,proto_bs['fH_SDS_H2O'].values[0],marker='D',c=color,edgecolors='none',s=15,zorder=5)
    if len(ana_bs)>0:
        ana_bs_m=ana_bs.merge(antimicrobial[['short_name']],on='short_name',how='inner')
        if len(ana_bs_m)>0:
            q1,med,q3=np.percentile(ana_bs_m['fH_SDS_H2O'],[25,50,75])
            ax.fill_between([i-0.25,i+0.25],q1,q3,color=color,alpha=0.2,zorder=2)
            ax.plot([i-0.25,i+0.25],[med,med],color=color,lw=1.0,zorder=4)
            jit=np.random.RandomState(hash(fam)%1000).normal(0,0.05,len(ana_bs_m))
            for j,(_,row) in enumerate(ana_bs_m.iterrows()):
                if row['short_name'] in LEADS_SET:
                    ax.scatter(i+jit[j],row['fH_SDS_H2O'],c=color,s=10,edgecolors='black',linewidth=0.5,zorder=5)
                    annotate_lead(ax,i+jit[j],row['fH_SDS_H2O'],LEADS_SHORT[row['short_name']],(8,6))
                else:
                    ax.scatter(i+jit[j],row['fH_SDS_H2O'],c=color,s=6,marker='o',edgecolors='none',linewidth=0.15,alpha=0.6,zorder=3)
ax.set_xticks(range(len(INACTIVE_ORDER)))
xlabs=ax.set_xticklabels(INACTIVE_ORDER,fontsize=6,rotation=20,ha='right')
for xl,fam in zip(xlabs,INACTIVE_ORDER): xl.set_color(FC[fam])
ax.set_ylabel('$\\alpha$-Helix in SDS (%)',fontsize=6); ax.set_ylim(-2,105)

# C
for i,fam in enumerate(INACTIVE_ORDER):
    ax=fig.add_subplot(gs_d[i])
    if i==0: ax.text(-0.25,1.10,'C',transform=ax.transAxes,**L)
    sns.kdeplot(data=ref_amp_props,x='charge',y='hydrophobicity',levels=5,color='gray',linewidths=0.3,alpha=0.4,ax=ax)
    ax.set_xlabel(''); ax.set_ylabel('')
    ana=antimicrobial[antimicrobial['family']==fam].dropna(subset=['SW'])
    if len(ana)>0:
        nl=ana[~ana['short_name'].isin(LEADS_SET)]
        if len(nl)>0:
            ax.scatter(nl['net_charge'],nl['mean_hydrophobicity'],c=nl['SW'].clip(0.1,10000),cmap=sw_cmap,norm=sw_norm,
                       marker='o',s=14,edgecolor='none',linewidth=0.2,alpha=0.85,zorder=3)
        for _,row in ana[ana['short_name'].isin(LEADS_SET)].iterrows():
            ax.scatter(row['net_charge'],row['mean_hydrophobicity'],c=[min(10000,max(0.1,row['SW']))],
                       cmap=sw_cmap,norm=sw_norm,marker='o',s=14,edgecolors='black',linewidth=0.5,zorder=5)
            annotate_lead(ax,row['net_charge'],row['mean_hydrophobicity'],LEADS_SHORT[row['short_name']],(6,8))
    proto=proto_by_fam.get(fam)
    if proto is not None:
        psw=proto.get('SW',np.nan)
        if pd.notna(psw): ax.scatter(proto['net_charge'],proto['mean_hydrophobicity'],c=[min(10000,max(0.1,psw))],cmap=sw_cmap,norm=sw_norm,marker='D',edgecolors='none',s=15,zorder=5)
        else: ax.scatter(proto['net_charge'],proto['mean_hydrophobicity'],marker='D',c=FC[fam],edgecolors='none',s=15,zorder=5)
    ax.set_title(fam,fontsize=6,color=FC[fam],pad=2); ax.set_xlim(-2,14); ax.set_ylim(-1.0,0.8)
    if i==0: ax.set_ylabel('Hydrophobicity',fontsize=6)
    else: ax.set_yticklabels([])
fig.text(0.475,0.590,'Net charge',ha='center',va='top',fontsize=6)
cbar_ax=fig.add_axes([0.92,0.610,0.012,0.165])
cb=plt.colorbar(ScalarMappable(norm=sw_norm,cmap=sw_cmap),cax=cbar_ax)
cb.set_label('Safety Window\n(HC$_{50}$/MIC$_{50}$)',fontsize=6)
cb.set_ticks([0.1,1,10,100,1000,10000]); cb.set_ticklabels(['0.1','1','10','100','1K','10K']); cb.ax.tick_params(labelsize=6)

# D: strips with per-lead offsets to avoid overlap
# NPN: BoCo1-5(38) BoCo1-9(38) same row; Mammutin-4(41) GQ20-4(43) nearby
# DiSC: more spread but BoCo1-5(96) BoCo1-9(78) same row
LEAD_OFFSETS_D6={
    '\u03a9-AT-BoCo1-5':  {'NPN':(10,-14),'DiSC':(10,-12)},
    '\u03a9-AT-BoCo1-9':  {'NPN':(10,14), 'DiSC':(10,12)},
    '\u03a9-AU-GQ20-4':   {'NPN':(10,-12),'DiSC':(10,-12)},
    '\u03a9-AP-Mammutin-1-4':{'NPN':(10,12),'DiSC':(10,12)},
}
mech=antimicrobial.dropna(subset=['NPN','DiSC']).copy()
npn_med=df.dropna(subset=['NPN'])['NPN'].median(); disc_med=df.dropna(subset=['DiSC'])['DiSC'].median()
dn_mech=df[df['category']=='de_novo'].dropna(subset=['NPN','DiSC'])
fam_order_e=INACTIVE_ORDER[::-1]
for pi,(metric,med_val,xlabel_text) in enumerate([
    ('NPN',npn_med,'Outer membrane permeabilization\nNPN MaxRel (%)'),
    ('DiSC',disc_med,'Cytoplasmic membrane depolarization\nDiSC$_3$(5) MaxRel (%)')
]):
    ax=fig.add_subplot(gs_e[pi])
    if pi==0: ax.text(-0.24,1.06,'D',transform=ax.transAxes,**L)
    ax.boxplot([dn_mech[metric].values],positions=[len(fam_order_e)],vert=False,widths=0.45,patch_artist=True,zorder=2,
               boxprops=dict(facecolor='#f0f0f0',edgecolor='none',linewidth=0.5),
               whiskerprops=dict(color='#999',linewidth=0.5),medianprops=dict(color='#555',linewidth=0.8),
               capprops=dict(color='#999',linewidth=0.5),flierprops=dict(marker='.',markerfacecolor='#ccc',markersize=1.5))
    for i,fam in enumerate(fam_order_e):
        fd=mech[mech['family']==fam]; color=FC[fam]
        if len(fd)==0: continue
        proto=proto_by_fam.get(fam)
        pr=df[df['short_name']==(proto['short_name'] if proto is not None else '')].dropna(subset=[metric])
        if len(pr)>0: ax.scatter(pr[metric].values,[i],marker='D',c=color,edgecolors='none',s=15,zorder=5)
        jit=np.random.RandomState(i+300+pi).normal(0,0.10,len(fd))
        for j,(idx,row) in enumerate(fd.iterrows()):
            if row['short_name'] in LEADS_SET:
                ax.scatter(row[metric],i+jit[j],c=color,s=14,edgecolors='black',linewidth=0.5,zorder=5)
                off=LEAD_OFFSETS_D6.get(row['short_name'],{}).get(metric,(10,10))
                annotate_lead(ax,row[metric],i+jit[j],LEADS_SHORT[row['short_name']],off)
            else:
                ax.scatter(row[metric],i+jit[j],c=color,s=12,marker='o',edgecolors='none',linewidth=0.2,alpha=0.7,zorder=3)
    ax.axvline(med_val,color='#ccc',ls=':',lw=0.5,zorder=1)
    ax.set_yticks(list(range(len(fam_order_e)))+[len(fam_order_e)])
    yl=ax.set_yticklabels(fam_order_e+['$\\it{De\\;novo}$'],fontsize=6)
    for y,f in zip(yl[:-1],fam_order_e): y.set_color(FC[f])
    yl[-1].set_color('#999'); ax.set_xlabel(xlabel_text,fontsize=6)
    fam_list=[f for f in INACTIVE_ORDER if len(mech[mech['family']==f])>0]
    groups=[mech[mech['family']==f][metric].dropna().values for f in fam_list]
    groups=[g for g in groups if len(g)>0]
    if len(groups)>=2:
        all_v=np.concatenate(groups); gm=all_v.mean()
        eta2=sum(len(g)*(g.mean()-gm)**2 for g in groups)/((all_v-gm)**2).sum()
        ax.text(0.98,0.02,f'$\\eta^2$ = {eta2:.2f}',transform=ax.transAxes,fontsize=6,ha='right',va='bottom',fontweight='bold',color='#555')

# E: heatmap (full lead names in y-tick labels, bold)
ax=fig.add_subplot(gs_b[0])
fig.text(0.015,0.340,'E',fontsize=8,fontweight='bold',va='top',ha='left')
strain_labels_ordered=[STRAIN_LABELS.get(s,s[:20]) for s in strain_order]
hm_rows,hm_labels,hm_colors,hm_peptides=[],[],[],[]
for fam in INACTIVE_ORDER:
    proto=proto_by_fam.get(fam); ana=antimicrobial[antimicrobial['family']==fam]
    if proto is not None:
        pr=mic[mic['short_name']==proto['short_name']]
        if len(pr)>0:
            hm_rows.append([float(pr[s].values[0]) for s in strain_order])
            hm_labels.append(fam); hm_colors.append(FC[fam]); hm_peptides.append(proto['short_name'])
    SELECTED_LEADS={'BoCo1':'\u03a9-AT-BoCo1-5','GQ20':'\u03a9-AU-GQ20-4','Mammutin-1':'\u03a9-AP-Mammutin-1-4'}
    if len(ana)>0:
        if fam in SELECTED_LEADS:
            bn=SELECTED_LEADS[fam]; br_=ana[ana['short_name']==bn]
            best=br_.iloc[0] if len(br_)>0 else ana.sort_values(['n_strains_le2','mic50'],ascending=[False,True]).iloc[0]
        else: best=ana.sort_values(['n_strains_le2','mic50'],ascending=[False,True]).iloc[0]
        br=mic[mic['short_name']==best['short_name']]
        if len(br)>0:
            hm_rows.append([float(br[s].values[0]) for s in strain_order])
            hm_labels.append(best['short_name']); hm_colors.append(FC[fam]); hm_peptides.append(best['short_name'])
hm_array=np.array(hm_rows); hm_array=np.where(np.isnan(hm_array),128,hm_array)
mic_cmap_rb=LinearSegmentedColormap.from_list('mic_rb',[(0.0,'#b2182b'),(0.25,'#ef8a62'),(0.5,'#f7f7f7'),(0.75,'#67a9cf'),(1.0,'#2166ac')])
mic_norm_hm=TwoSlopeNorm(vmin=0,vcenter=32,vmax=64)
hm_display=hm_array.clip(0,64)
im=ax.imshow(hm_display,aspect='auto',cmap=mic_cmap_rb,norm=mic_norm_hm,interpolation='nearest')
for yi in range(hm_array.shape[0]):
    for xi in range(hm_array.shape[1]):
        val=hm_array[yi,xi]
        if val>64: continue
        normed=mic_norm_hm(min(val,64)); tc='white' if normed<0.15 or normed>0.85 else 'black'
        ax.text(xi,yi,f'{val:.1f}' if val<1 else f'{val:.0f}',ha='center',va='center',fontsize=6,color=tc)
ax.set_xticks(range(len(strain_labels_ordered))); ax.tick_params(axis='x',which='major',pad=2,length=0)
xlabels_hm=ax.set_xticklabels(strain_labels_ordered,rotation=45,ha='right',fontsize=6)
for xi,s in enumerate(strain_order):
    xlabels_hm[xi].set_color('#DC2626' if STRAIN_GRAM[s]=='+' else '#2563EB')
    if STRAIN_MDR[s]: xlabels_hm[xi].set_fontweight('bold')
ax.set_yticks(range(len(hm_labels))); ylhm=ax.set_yticklabels(hm_labels,fontsize=6)
for yi_l,(yl,c) in enumerate(zip(ylhm,hm_colors)):
    yl.set_color(c)
    if hm_peptides[yi_l] in LEADS_SET: yl.set_fontweight('bold')  # Bold only in heatmap y-ticks
for k in range(1,len(INACTIVE_ORDER)): ax.axhline(k*2-0.5,color='white',lw=1.5)
mic_cb_ax=fig.add_axes([0.865,0.215,0.015,0.100])
cb_mic=plt.colorbar(im,cax=mic_cb_ax); cb_mic.set_label('MIC ($\\mu$M)',fontsize=6,labelpad=3)
cb_mic.set_ticks([0,8,16,32,64]); cb_mic.ax.tick_params(labelsize=6); cb_mic.outline.set_visible(False)

# Toxicity columns
n_strains=len(strain_order)
cc50_dict=dict(zip(cc50['short_name'],cc50['CC50'].clip(0.1,128)))
hc50_dict=dict(zip(hc50['short_name'],hc50['HC50'].clip(0.1,128)))
def fmt_tox(val):
    if pd.isna(val): return '--'
    if val>=128: return '>128'
    if val>10: return f'{val:.0f}'
    return f'{val:.1f}'
tox_cmap=LinearSegmentedColormap.from_list('tox_gr',[(0.0,'#b2182b'),(0.2,'#ef8a62'),(0.4,'#fddbc7'),(0.5,'#f7f7f7'),(0.6,'#a6d96a'),(0.8,'#1a9641'),(1.0,'#006837')])
tox_norm=LogNorm(vmin=0.5,vmax=128)
def tox_color(val):
    if pd.isna(val): return '#999'
    return tox_cmap(tox_norm(np.clip(val,0.5,128)))
for yi,pep in enumerate(hm_peptides):
    hv=hc50_dict.get(pep,np.nan); cv=cc50_dict.get(pep,np.nan)
    for xp,val in [(n_strains+0.5,hv),(n_strains+1.5,cv)]:
        c=tox_color(val); ax.add_patch(Rectangle((xp-0.45,yi-0.5),0.9,1.0,facecolor=c,edgecolor='white',linewidth=0.5,clip_on=False,zorder=2))
    for xp,val in [(n_strains+0.5,hv),(n_strains+1.5,cv)]:
        txt=fmt_tox(val); c_rgba=tox_color(val)
        lum=0.299*c_rgba[0]+0.587*c_rgba[1]+0.114*c_rgba[2] if not pd.isna(val) else 0.5
        ax.text(xp,yi,txt,ha='center',va='center',fontsize=6,color='white' if lum<0.55 else '#333',zorder=3,clip_on=False)
ax.text(n_strains+0.5,-0.7,'HC$_{50}$',ha='center',va='bottom',fontsize=6,color='#333',fontweight='bold')
ax.text(n_strains+1.5,-0.7,'CC$_{50}$',ha='center',va='bottom',fontsize=6,color='#333',fontweight='bold')
tox_cb_ax=fig.add_axes([0.940,0.215,0.015,0.100])
cb_tox=plt.colorbar(ScalarMappable(norm=tox_norm,cmap=tox_cmap),cax=tox_cb_ax)
cb_tox.set_label('CC$_{50}$/HC$_{50}$\n($\\mu$M)',fontsize=6,labelpad=2)
cb_tox.set_ticks([1,10,100]); cb_tox.set_ticklabels(['1','10','$\\geq$128']); cb_tox.ax.tick_params(labelsize=6); cb_tox.outline.set_visible(False)

plt.savefig('/home/pszymczak/code/omegamp-dashboard/figures/figure5_analog.pdf',bbox_inches='tight')
plt.savefig('/home/pszymczak/code/omegamp-dashboard/figures/figure5_analog.svg',bbox_inches='tight')
plt.savefig('/home/pszymczak/code/omegamp-dashboard/figures/figure5_analog.png',dpi=600,bbox_inches='tight')
plt.show()
print("Saved figure5_analog")